# Model V2 - Hourly Electricity Price Prediction

This notebook turns the engineered feature table into the first real forecasting model.

Goal: compare a simple time-series baseline with an XGBoost model that uses lag, rolling, calendar, holiday, and weather features.

## Notebook Plan

1. Load the engineered feature table.
2. Check time order, missing values, and target availability.
3. Split by time, not by shuffle.
4. Train a naive baseline for comparison.
5. Train the V2 XGBoost model.
6. Evaluate, inspect errors, and save results.

In [ ]:
# Imports: standard libraries and ML tools.
# Comments: these provide file paths, plotting, numeric arrays, and model/metrics.
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# Display settings for pandas so we can view many columns comfortably.
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)

## 1. Load Data

Use the engineered dataset created by `feature_engineering.ipynb`. This is the main input for V2.

In [ ]:
# Load engineered features: this CSV is produced by feature_engineering.ipynb.
# It contains the target 'price', timestamp 'datetime', and many engineered feature columns.
data_path = Path('../data/convertData/finland_electricity_features_v2.csv')
df = pd.read_csv(data_path, parse_dates=['datetime'])
# Ensure rows are sorted in time order (important for time-series models).
df = df.sort_values('datetime').reset_index(drop=True)

print('Shape:', df.shape)
print('Date range:', df['datetime'].min(), '->', df['datetime'].max())
print('Duplicated timestamps:', df['datetime'].duplicated().sum())
df.head(3)

## 2. Target and Feature Check

Keep only columns that are available before prediction time. Remove the target and timestamp from the model input.

In [ ]:
# Define target and features.
# target_col is the value we want to predict (price).
# time_col is the timestamp column used for sorting and splitting.
target_col = 'price'
time_col = 'datetime'
exclude_cols = {target_col, time_col}

# feature_cols contains all columns except the target and time columns.
feature_cols = [col for col in df.columns if col not in exclude_cols]
# model_df is a copy where first columns are time and target followed by features.
model_df = df[[time_col, target_col] + feature_cols].copy()

print('Number of features:', len(feature_cols))
print('Feature sample (first 15):', feature_cols[:15])
print('Missing values per column (top):')
display(model_df.isna().sum().loc[lambda s: s > 0].sort_values(ascending=False).head(20))

## 3. Time-Based Split

Use chronological splitting so the model only learns from the past and is tested on the future.

In [ ]:
# Drop rows with missing values (simple strategy). Advanced: impute where needed.
model_df = model_df.dropna().reset_index(drop=True)

# For clarity (and to match V1), use an 80/20 chronological split: first 80% train, last 20% test.
n_rows = len(model_df)
train_end = int(n_rows * 0.80)  # 80% for training

train_df = model_df.iloc[:train_end].copy()
test_df = model_df.iloc[train_end:].copy()

# Prepare feature / target matrices for model training and testing.
X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]

print('Train:', train_df[time_col].min(), '->', train_df[time_col].max(), X_train.shape)
print('Test :', test_df[time_col].min(), '->', test_df[time_col].max(), X_test.shape)

## 4. Naive Baseline

A simple seasonal baseline helps us see whether the model is really learning something useful.

In [ ]:
# Naive baselines: simple forecasts that use previous prices.
# 24-hour lag: predict the current price equal to the price 24 hours ago.
naive_24h = X_test['price_lag_24h']
# 168-hour (1 week) lag: predict equal to same hour last week.
naive_168h = X_test['price_lag_168h']

def evaluate(y_true, y_pred, name):
    # Return common regression metrics in a dict.
    return {
        'model': name,
        'mae': mean_absolute_error(y_true, y_pred),
        'mse': mean_squared_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'r2': r2_score(y_true, y_pred),
    }

baseline_results = pd.DataFrame([
    evaluate(y_test, naive_24h, 'Naive_24h'),
    evaluate(y_test, naive_168h, 'Naive_168h'),
])
baseline_results

## 5. Train XGBoost Model

This is the main V2 model. It uses the full engineered feature table instead of raw weather only.

In [ ]:
# Instantiate XGBoost model with reasonable defaults.
# These are hyperparameters — they control model complexity and learning.
model_v2 = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=200,  # number of trees
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

# Train on the training set only (no validation here to match V1 style).
model_v2.fit(X_train, y_train)

print('Training finished.')

## 6. Evaluate Model

Compare the engineered-feature model against the naive baseline on the holdout test set.

In [ ]:
# Make predictions on the test set and evaluate metrics (same style as V1).
test_pred = model_v2.predict(X_test)

# Compute metrics for XGBoost on the test set.
mse_score = mean_squared_error(y_true=y_test, y_pred=test_pred)
rmse_score = np.sqrt(mse_score)
mae_score = mean_absolute_error(y_true=y_test, y_pred=test_pred)
r2_score_val = r2_score(y_true=y_test, y_pred=test_pred)
n = len(X_test)
k = X_test.shape[1]
adjusted_r2 = 1 - (1 - r2_score_val) * (n - 1) / (n - k - 1)

print('================ V2 XGBoost Evaluation ================')
print('Mean Absolute Error (MAE)            :', mae_score)
print('Mean Squared Error (MSE)             :', mse_score)
print('Root Mean Squared Error (RMSE)       :', rmse_score)
print('R2 Score (R2)                        :', r2_score_val)
print('Adjusted R2                           :', adjusted_r2)
print('=======================================================')

# Combine baseline and model metrics for easy comparison.
results = pd.DataFrame([evaluate(y_test, test_pred, 'XGB_V2_Test')])
comparison = pd.concat([baseline_results, results], ignore_index=True)
comparison.sort_values('rmse')

## 7. Feature Importance and Error Check

Look at the most useful features and inspect the prediction residuals.

In [ ]:
# Feature importance: which features the XGBoost model used most.
# This helps understand what drives predictions (global explanation).
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': model_v2.feature_importances_
}).sort_values('importance', ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
importance_df.sort_values('importance').plot.barh(x='feature', y='importance', ax=axes[0], legend=False, color='#2a6fdb')
axes[0].set_title('Top Feature Importance')
axes[0].set_xlabel('Importance')
axes[0].set_ylabel('Feature')

# Residuals vs predictions plot helps spot biases and outliers (error check).
residuals = y_test - test_pred
axes[1].scatter(test_pred, residuals, s=10, alpha=0.4, color='#d95f02')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_title('Residuals vs Prediction')
axes[1].set_xlabel('Predicted price')
axes[1].set_ylabel('Residual')

plt.tight_layout()
plt.show()

importance_df

## 8. Save Outputs

Save predictions and the fitted model so the next notebook or script can reuse them.

In [ ]:
# Save outputs: predictions CSV, trained model, and feature list.
# The CSV contains: datetime, actual price, predicted_price, residual.
# Why save? So you can inspect errors, make plots later, or serve the model.
output_dir = Path('../reports/modelV2')
output_dir.mkdir(parents=True, exist_ok=True)

test_output = test_df[[time_col, target_col]].copy()
test_output['predicted_price'] = test_pred
test_output['residual'] = test_output[target_col] - test_output['predicted_price']
test_output.to_csv(output_dir / 'test_predictions.csv', index=False)
# Save trained model for later reuse in a predict script.
joblib.dump(model_v2, output_dir / 'xgb_model_v2.pkl')
# Save the feature column list so training and serving use the same order.
pd.Series(feature_cols, name='feature').to_csv(output_dir / 'feature_columns.csv', index=False)

print('Saved files to:', output_dir.resolve())
print('Test prediction rows:', len(test_output))

## 9. Next Steps

If V2 improves over V1, the next upgrade can add cross-validation, SHAP explanations, or a prediction script.